In [1]:
#!/usr/bin/env python3
# coding: utf-8

import os
import csv
import cloudinary
import cloudinary.uploader
import firebase_admin
from firebase_admin import credentials, firestore
from rapidfuzz import fuzz
import unicodedata
import re

In [2]:
# ========== CONFIG ========== #
CREDENTIALS_PATH = '../../private_key.json'  # caminho para sua chave de serviço
IMAGES_FOLDER = r'C:\Users\Layanny\Documents\patrimonygo\src\assets\Patrimonios'
CLOUD_NAME = 'dglfvvzg1'
CLOUD_API_KEY = '555873354438444'
CLOUD_API_SECRET = '_ELGOWW6c3l9-e9-94H_GrhoSfk'
BUCKET_FOLDER = 'patrimonios'  # pasta dentro do Cloudinary
FIRESTORE_COLLECTION = 'patrimonios_santos'
FUZZY_THRESHOLD = 80  # 0-100, ajuste se quiser aceitar menos
UPDATE_APPEND = True   # True => concatena URLs ao campo imageUrls existente; False => substitui
UNMATCHED_CSV = 'unmatched.csv'
# ============================ #

In [3]:
# inicializa cloudinary
cloudinary.config(
    cloud_name=CLOUD_NAME,
    api_key=CLOUD_API_KEY,
    api_secret=CLOUD_API_SECRET,
    secure=True
)

In [4]:
cred = credentials.Certificate(CREDENTIALS_PATH)
firebase_admin.initialize_app(cred)
db = firestore.client()

In [5]:
# normalização do nome: remove acentos, pontuação, multiple spaces, lower
def normalize(text: str) -> str:
    if text is None:
        return ''
    # remove extensão se por acaso
    text = re.sub(r'\.[A-Za-z0-9]+$', '', text)
    # normalize unicode (remove acentos)
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    # remove non-alphanumeric (mas manter espaços)
    text = re.sub(r'[^0-9A-Za-z\s]', '', text)
    # squeeze spaces and lower
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

In [6]:
def group_images(folder):
    groups = {}
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.gif')):
            continue
        # remove ext and trailing number: split by last space
        # ex: "BASILICA EMBARE 02.jpg" -> base_raw = "BASILICA EMBARE"
        base_raw = fname.rsplit(' ', 1)[0]
        # if the last token is not a number, try removing numeric in parentheses or last token
        # fallback: use entire name without extension
        if base_raw == fname:
            base_raw = os.path.splitext(fname)[0]
        groups.setdefault(base_raw, []).append(fname)
    return groups



In [7]:
def upload_image_to_cloudinary(path, public_id=None):
    # public_id optional; Cloudinary will generate id if None
    try:
        res = cloudinary.uploader.upload(
            path,
            folder=BUCKET_FOLDER,
            public_id=public_id,
            overwrite=False,
            resource_type="image"
        )
        return res.get('secure_url')
    except Exception as e:
        print(f'  ❌ Erro upload {path}: {e}')
        return None


In [8]:
def build_firestore_index():
    docs = list(db.collection(FIRESTORE_COLLECTION).stream())
    index = {}
    names = {}
    for d in docs:
        data = d.to_dict()
        name = data.get('name') or data.get('title') or ''
        norm = normalize(name)
        index[norm] = d.id
        names[norm] = name
    return index, names

In [9]:
def main():
    groups = group_images(IMAGES_FOLDER)
    print(f'Grupos encontrados: {len(groups)}')

    index, names_map = build_firestore_index()
    print(f'Documentos no Firestore indexados: {len(index)}')

    unmatched_rows = []
    updated_count = 0

    for base_raw, files in groups.items():
        print(f'\n📍 Processando grupo: "{base_raw}" ({len(files)} arquivos)')
        normalized_base = normalize(base_raw)

        # tentativa de correspondência exata
        matched_doc_id = index.get(normalized_base)
        matched_name = names_map.get(normalized_base)

        # se não exato, faz fuzzy match
        if not matched_doc_id:
            best_score = 0
            best_norm = None
            for norm_name in index.keys():
                score = fuzz.ratio(normalized_base, norm_name)
                if score > best_score:
                    best_score = score
                    best_norm = norm_name
            if best_score >= FUZZY_THRESHOLD:
                matched_doc_id = index.get(best_norm)
                matched_name = names_map.get(best_norm)
                print(f'  🔎 Fuzzy matched "{base_raw}" -> "{matched_name}" (score {best_score})')
            else:
                print(f'  ⚠️ Nenhuma correspondência confiável (melhor score {best_score}). Será registrado como unmatched.')
        
        # faz upload de imagens e coleta URLs
        urls = []
        for f in files:
            full_path = os.path.join(IMAGES_FOLDER, f)
            print(f'  -> upload {f} ...', end=' ')
            url = upload_image_to_cloudinary(full_path)
            if url:
                urls.append(url)
                print('OK')
            else:
                print('FALHOU')

        # atualiza firestore se encontrou doc
        if matched_doc_id:
            doc_ref = db.collection(FIRESTORE_COLLECTION).document(matched_doc_id)
            try:
                if UPDATE_APPEND:
                    doc_snapshot = doc_ref.get()
                    existing = doc_snapshot.to_dict().get('imageUrls') or []
                    # evita duplicatas
                    new_urls = existing + [u for u in urls if u not in existing]
                    doc_ref.update({'imageUrls': new_urls})
                else:
                    doc_ref.update({'imageUrls': urls})
                print(f'  🔁 Firestore atualizado: {matched_doc_id} (nome registrado: "{matched_name}")')
                updated_count += 1
            except Exception as e:
                print(f'  ❌ Erro ao atualizar Firestore para {matched_doc_id}: {e}')
                unmatched_rows.append((base_raw, ','.join(files), 'update_error', str(e)))
        else:
            unmatched_rows.append((base_raw, ','.join(files), 'no_match', ''))
    
    # salvar unmatched em CSV para revisão manual
    if unmatched_rows:
        with open(UNMATCHED_CSV, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['base_name', 'files', 'reason', 'detail'])
            writer.writerows(unmatched_rows)
        print(f'\n⚠️ Há {len(unmatched_rows)} grupos não correspondidos. Veja {UNMATCHED_CSV}')
    print(f'\n✔️ Finalizado. Documentos atualizados: {updated_count}')

if __name__ == '__main__':
    main()

Grupos encontrados: 25
Documentos no Firestore indexados: 1

📍 Processando grupo: "ALFÂNDEGA DA RECEITA FEDERAL DO BRASIL EM SANTOS" (3 arquivos)
  ⚠️ Nenhuma correspondência confiável (melhor score 0). Será registrado como unmatched.
  -> upload ALFÂNDEGA DA RECEITA FEDERAL DO BRASIL EM SANTOS 01.jpg ... OK
  -> upload ALFÂNDEGA DA RECEITA FEDERAL DO BRASIL EM SANTOS 01.png ... OK
  -> upload ALFÂNDEGA DA RECEITA FEDERAL DO BRASIL EM SANTOS 03.jpg ... OK

📍 Processando grupo: "BASÍLICA EMBARÉ" (3 arquivos)
  ⚠️ Nenhuma correspondência confiável (melhor score 0). Será registrado como unmatched.
  -> upload BASÍLICA EMBARÉ 01.jpg ... OK
  -> upload BASÍLICA EMBARÉ 02.jpg ... OK
  -> upload BASÍLICA EMBARÉ 03.jpg ... OK

📍 Processando grupo: "BONDES - LINHA TURISTICA" (2 arquivos)
  ⚠️ Nenhuma correspondência confiável (melhor score 0). Será registrado como unmatched.
  -> upload BONDES - LINHA TURISTICA 01.png ... OK
  -> upload BONDES - LINHA TURISTICA 02.png ... OK

📍 Processando grup

In [10]:
# diagnostic_matches.py
import os
import csv
import unicodedata
import re
from rapidfuzz import fuzz
import firebase_admin
from firebase_admin import credentials, firestore

# === CONFIGURE AQUI ===
CREDENTIALS_PATH = 'firebase-key.json'
IMAGES_FOLDER = r'C:\Users\Layanny\Documents\patrimonygo\src\assets\Patrimonios'
FIRESTORE_COLLECTION = 'patrimonios_santos'
TOP_N = 5  # quantas sugestões mostrar por grupo
# ======================

# init firebase
cred = credentials.Certificate(CREDENTIALS_PATH)
firebase_admin.initialize_app(cred)
db = firestore.client()

def normalize(text: str) -> str:
    if text is None:
        return ''
    text = re.sub(r'\.[A-Za-z0-9]+$', '', text)      # remove extensão se houver
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    text = re.sub(r'[^0-9A-Za-z\s]', '', text)       # remove pontuação (mantém espaços)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

def group_images(folder):
    groups = {}
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.gif')):
            continue
        # tentativa de extrair base: remove última parte que costuma ser número
        # Ex: "ALFÂNDEGA DA ... 01.jpg" -> "ALFÂNDEGA DA ..."
        base_raw = fname.rsplit(' ', 1)[0]
        if base_raw == fname:
            base_raw = os.path.splitext(fname)[0]
        groups.setdefault(base_raw, []).append(fname)
    return groups

def build_index():
    docs = list(db.collection(FIRESTORE_COLLECTION).stream())
    index = {}
    for d in docs:
        data = d.to_dict()
        name = data.get('name') or data.get('title') or ''
        norm = normalize(name)
        # se houver duplicatas normalizadas, guardamos lista (defensivo)
        index.setdefault(norm, []).append((d.id, name))
    return index

def list_firestore_names():
    docs = list(db.collection(FIRESTORE_COLLECTION).stream())
    names = []
    for d in docs:
        data = d.to_dict()
        name = data.get('name') or data.get('title') or ''
        names.append((d.id, name, normalize(name)))
    return names

def diagnostic():
    groups = group_images(IMAGES_FOLDER)
    print(f'Grupos detectados: {len(groups)}')
    fs_names = list_firestore_names()
    fs_norms = [n for (_,_,n) in fs_names]

    out_rows = []
    for base_raw, files in groups.items():
        base_norm = normalize(base_raw)
        # compute scores against every firestore normalized name
        scores = []
        for doc_id, orig_name, norm_name in fs_names:
            score = fuzz.ratio(base_norm, norm_name)
            scores.append((score, doc_id, orig_name, norm_name))
        scores.sort(reverse=True, key=lambda x: x[0])
        best = scores[:TOP_N]
        print('\n----------------------------------------')
        print(f'Grupo de arquivo: "{base_raw}"  (normalizado: "{base_norm}")\nArquivos: {files}')
        print('Melhores correspondências (score, doc_id, firestore_name):')
        for score, doc_id, orig_name, norm_name in best:
            print(f'  {score:3d}  {doc_id}  "{orig_name}"  (norm: "{norm_name}")')
            out_rows.append([base_raw, ','.join(files), base_norm, score, doc_id, orig_name, norm_name])
    # salvar CSV com sugestões
    with open('suggestions.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['group_base', 'files', 'group_norm', 'score', 'doc_id', 'firestore_name', 'firestore_norm'])
        writer.writerows(out_rows)
    print('\nDone. Arquivo suggestions.csv gerado — abra e revise as sugestões.')
    print('Se quiser, podemos: baixar matches com score>=X automaticamente, reduzir threshold, ou criar um mapeamento manual (CSV) para aplicar atualizações.')
    
if __name__ == '__main__':
    diagnostic()


FileNotFoundError: [Errno 2] No such file or directory: 'firebase-key.json'